# 03 · Data Cleaning
**Goal:** Fix all data-quality problems (including what `02_data_modeling.ipynb` has found), so that `04_data_integration.ipynb` can merge the four tables without data errors.

## Open decisions (not yet made — decide here, then implement)

- [x] **Menu dedup rule.** Menu has 242 rows but only 106 unique `(product_name, size)` keys, and 24 of those groups have genuinely different `calories` / `price_usd` / `sugar_g` / `caffeine_mg` values (see `02_data_modeling.ipynb`, the Caffè Latte Grande example ranges \$3.99–\$4.49 and 70–240 calories). **This doesn't block the master dataset join** — per `02_data_modeling.ipynb`'s decision, `Transaction` already carries its own `base_price`/`calories`/`sugar_g`/`caffeine_mg`, so the only field the join actually needs from `Menu` is `seasonal_flag`. Dedup still needs to happen (one row per key before `Menu` can be a dimension table), but the aggregation rule only has to produce a trustworthy `seasonal_flag` per group — the conflicting price/nutrition values don't need to be reconciled for the join to work. **Decision (Step 4.2):** median per group for `price_usd`/`calories`/`sugar_g`/`caffeine_mg`, first-row value for `category`, majority vote (tie → True) for `seasonal_flag`.
- [ ] **2026-03 Macro gap.** Macro data ends `2026-02-01`; 122 transactions fall in `2026-03` with no matching macro month. Decide: leave `cpi`/`avg_hourly_earnings`/`real_wage_index` NULL for those rows, forward-fill the last known macro values, or drop the 122 rows.
- [ ] **Master dataset enrichment.** Decide whether to pull `weather.temp_max_f` / `weather.temp_min_f` and/or `macro.avg_hourly_earnings` / `macro.real_wage_index` into the master dataset (currently only `temp_f` and `cpi` are embedded in Transaction). This affects the join in `04_data_integration.ipynb`, so decide it here before writing that notebook's join code.

### 1. Load Data

In [1]:
import pandas as pd

menu         = pd.read_csv('../data/raw/starbucks_menu.csv')
transactions = pd.read_csv('../data/raw/synthetic_transactions.csv')
macro        = pd.read_csv('../data/raw/fred_macro.csv')
weather      = pd.read_csv('../data/raw/weather_daily.csv')

### 2. Data Overview

Most data overview are already covered in `01_data_audit.ipynb`.
Additional step of checking duplicates is added below:

In [2]:
menu.duplicated().sum()

np.int64(1)

In [3]:
transactions.duplicated().sum()

np.int64(0)

In [4]:
macro.duplicated().sum()

np.int64(0)

In [5]:
weather.duplicated().sum()

np.int64(0)

### 3. Cleaning Missing Values
Assess missing values in each dataset; determine whether they need removal, imputation, or retained.

### Nulls that need to resolve
Discovered from `01_data_audit.ipynb`'s `.isnull().sum()` output:

- [x] `transactions.cpi` / `transactions.total_price` — 4,324 nulls each. Rows dropped (see Step 3.1).
- [x] `transactions.caffeine_mg` — 8,186 nulls. Not recoverable from `transactions` or `menu`; left NULL permanently — `menu`'s nutritional columns are never pulled into the master dataset anyway (see Step 3.2).
- [x] `menu.caffeine_mg` — 23 nulls. Deferred — resolves as a side effect of Menu dedup in Step 4 (see Step 3.3).
- [x] `macro.cpi` / `macro.real_wage_index` — 1 null each. Fixed via linear interpolation (see Step 3.4).

#### 3.1 `transactions.cpi` and `transactions.total_price`

Since `transactions.cpi` has 4,324 nulls and `transactions.total_price` also has 4,324 nulls, I will check whether they are the same rows.

In [6]:
transactions[transactions['cpi'].isnull() & transactions['total_price'].isnull()]

,transaction_id,date,hour,time_slot,city,persona,is_weekend,temp_f,cpi,category,product_name,size,base_price,customizations,n_customizations,upcharge,total_price,calories,sugar_g,caffeine_mg
30,TXN-000031,2025-09-30,11,late_morning,Chicago,student,0,70.3,NaN,Frappuccino® Blended Coffee,Coffee,Venti,5.43,none,0,0.0,NaN,310,69,95.0
48,TXN-000049,2025-09-06,7,morning_rush,New York,health_conscious,1,74.6,NaN,Coffee,Brewed Coffee,Grande,2.55,none,0,0.0,NaN,5,0,330.0
54,TXN-000055,2025-09-30,12,lunch,Houston,weekend_explorer,0,82.4,NaN,Frappuccino® Light Blended Coffee,Java Chip,Grande,4.59,none,0,0.0,NaN,220,39,105.0
93,TXN-000094,2025-09-06,12,lunch,New York,afternoon_treat,1,74.6,NaN,Frappuccino® Blended Coffee,Coffee,Grande,4.79,Vanilla Syrup|Oat Milk,2,1.3,NaN,180,36,70.0
157,TXN-000158,2025-09-18,11,late_morning,Los Angeles,weekend_explorer,0,73.9,NaN,Coffee,Brewed Coffee,Tall,2.18,none,0,0.0,NaN,4,0,260.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99913,TXN-099914,2025-09-11,8,morning_rush,Houston,health_conscious,0,82.1,NaN,Classic Espresso Drinks,Caffè Mocha (Without Whipped Cream),Tall,3.61,Caramel Drizzle,1,0.6,NaN,170,27,95.0
99928,TXN-099929,2025-09-02,15,afternoon,New York,weekend_explorer,0,69.9,NaN,Frappuccino® Light Blended Coffee,Mocha,Grande,4.47,Mocha Sauce,1,0.6,NaN,150,30,95.0
99952,TXN-099953,2025-09-09,16,afternoon,Houston,afternoon_treat,0,75.8,NaN,Signature Espresso Drinks,Caramel Macchiato,Venti,5.90,Oat Milk,1,0.7,NaN,240,41,150.0
99969,TXN-099970,2025-09-28,9,morning_rush,Chicago,student,1,70.1,NaN,Frappuccino® Blended Coffee,Coffee,Grande,4.79,Caramel Drizzle,1,0.6,NaN,180,36,70.0


Exactly 4324. So they are the same rows. This also means that the missing values of `cpi` and `total_price` might because of a same problem.

In [7]:
missing_rows = transactions[(transactions['cpi'].isnull()) & (transactions['total_price'].isnull()) ]

missing_rows['date'].unique()

array(['2025-09-30', '2025-09-06', '2025-09-18', '2025-09-19',
       '2025-09-10', '2025-09-07', '2025-09-08', '2025-09-23',
       '2025-09-28', '2025-09-25', '2025-09-29', '2025-09-04',
       '2025-09-12', '2025-09-09', '2025-09-03', '2025-09-13',
       '2025-09-17', '2025-09-27', '2025-09-24', '2025-09-15',
       '2025-09-22', '2025-09-02', '2025-10-01', '2025-09-11',
       '2025-09-05', '2025-09-20', '2025-09-16', '2025-09-21',
       '2025-09-14', '2025-09-26'], dtype=object)

The dates of those rows are almost entirely September 2025, plus one day in October. That's not the 2026-03 macro gap. check whether `Macro` itself is missing September 2025 data.

In [8]:
macro[macro['date'].str.startswith('2025-09')]

,date,cpi,avg_hourly_earnings,real_wage_index
54,2025-09-01,324.245,36.7,11.3186


`Macro` has a real September 2025 CPI value, so this isn't a macro coverage gap. The nulls in `Transactions` look like corrupted in the raw file itself.

The `cpi` value in those rows could be recovered by `cpi` in `Macro`, but `total_price` should also be recovered at the same time to make those rows be able to use.

Next step:
check whether `total_price` is recoverable.

`base_price` and `upcharge` are intact on every one of these rows, so check whether `total_price` is a fixed function of `(product_name, size, upcharge)`. If it is, the correct value could be looked up from other rows with the same combination:

In [9]:
transactions.groupby(['product_name', 'size', 'upcharge'])['total_price'].nunique().value_counts()

total_price
1     392
2     217
3     143
5     100
18     99
19     93
4      93
6      82
17     72
7      68
9      66
12     64
8      63
15     56
13     55
11     54
10     51
16     42
14     29
20     14
0      11
30      3
25      3
21      2
27      2
31      1
22      1
36      1
33      1
24      1
23      1
Name: count, dtype: int64

Most `(product_name, size, upcharge)` combinations have far more than one distinct `total_price`. Which means that pricing depends on factors beyond these three, so there's no reliable way to reconstruct the true value.

**Decision: drop all these 4,324 rows.** They're a small share of the dataset (~4.3%), but mostly concentrated in September 2025. Flag that in Step 9 to notify that that month's transaction count may be understated.

In [10]:
transactions = transactions[~(transactions['cpi'].isnull() & transactions['total_price'].isnull())].reset_index(drop=True)
#delete rows with NULL total_price from transactions.

transactions.shape

(95676, 20)

Finished cleaning nulls of `transactions.cpi` and `transactions.total_price`

#### 3.2 `transactions.caffeine_mg`

8,186 nulls remain. Check which products they belong to:

In [11]:
transactions[transactions['caffeine_mg'].isnull()]['product_name'].value_counts()

product_name
Tazo® Full-Leaf Tea Latte                              2134
Iced Brewed Coffee (With Milk & Classic Syrup)         1610
Banana Chocolate Smoothie                              1319
Shaken Iced Tazo® Tea Lemonade (With Classic Syrup)    1159
Shaken Iced Tazo® Tea (With Classic Syrup)              842
Tazo® Tea                                               790
Name: count, dtype: int64

Only 6 products, but check whether `caffeine_mg` is a fixed value per `(product_name, size)`

In [12]:
transactions.groupby(['product_name', 'size'])['caffeine_mg'].nunique().value_counts()

caffeine_mg
1    88
0    15
2     3
Name: count, dtype: int64

Most `(product_name, size)` groups have exactly 1 known `caffeine_mg` value (good, but they already have no nulls, so there's nothing to fill there);

15 groups have 0, meaning that every single row for that product/size is null, so there's no value anywhere in `transactions` to copy from (these 15 `(product_name, size)` are the source of those 8,186 nulls);

few groups have 2 different values with no nulls at all, which is a consistency problem for Step 6, not a missing-value problem.

Next:
Check whether `menu` has the value for the products that are null everywhere in `transactions`:

In [13]:
null_products = transactions[transactions['caffeine_mg'].isnull()]['product_name'].unique()
menu[menu['product_name'].isin(null_products)][['product_name', 'size', 'caffeine_mg']].sort_values(['product_name', 'size'])

,product_name,size,caffeine_mg
172,Banana Chocolate Smoothie,Grande,NaN
173,Banana Chocolate Smoothie,Grande,15.0
174,Banana Chocolate Smoothie,Grande,15.0
158,Iced Brewed Coffee (With Milk & Classic Syrup),Grande,NaN
159,Iced Brewed Coffee (With Milk & Classic Syrup),Grande,90.0
160,Iced Brewed Coffee (With Milk & Classic Syrup),Grande,90.0
161,Iced Brewed Coffee (With Milk & Classic Syrup),Grande,125.0
162,Iced Brewed Coffee (With Milk & Classic Syrup),Grande,125.0
164,Iced Brewed Coffee (With Milk & Classic Syrup),Grande,170.0
165,Iced Brewed Coffee (With Milk & Classic Syrup),Grande,170.0


`Menu` isn't a reliable source for these either: some of these products are entirely null in `Menu` too (e.g. `Tazo® Tea`), and the rest sit inside the messy duplicate-key groups flagged in the Menu dedup open decision.

But according to `02_data_modeling.ipynb`'s decision, `Menu`'s nutritional columns will not be pulled into the master dataset because `Transactions` already carries its own `base_price`/`calories`/`sugar_g`/`caffeine_mg`, and the only field the join needs from `Menu` is `seasonal_flag`. So there's no future step where `menu.caffeine_mg` would backfill `transactions.caffeine_mg`.

**Decision: leave `transactions.caffeine_mg` nulls as NULL.** It's a secondary nutritional field on `transactions` itself, not required for revenue/portfolio analysis, so the gap is low-risk to accept as-is.

#### 3.3 `menu.caffeine_mg`

These 23 nulls sit inside the duplicate-key groups covered by the Menu dedup open decision above, but they split into two very different cases:

**2 nulls** are in groups that also have other, non-null rows. When Step 4's dedup takes the median per `(product_name, size)`, `median()` skips NaN automatically. These resolve for free, no separate fix needed.

**21 nulls** are in groups where every row is null. Since median of an all-NaN group is still NaN, dedup can't rescue these. This is the same underlying gap as `transactions.caffeine_mg` in Step 3.2, which is that these tea products simply never had caffeine recorded anywhere in the raw data. They'll stay NULL after Step 4.

#### 3.4 `macro.cpi` / `macro.real_wage_index`

In [14]:
macro[macro['cpi'].isnull() | macro['real_wage_index'].isnull()]

,date,cpi,avg_hourly_earnings,real_wage_index
55,2025-10-01,NaN,36.85,NaN


Both nulls are on the same row (`2025-10-01`) — `avg_hourly_earnings` is present there, only `cpi` and `real_wage_index` are missing. These are smooth, slow-moving monthly indicators (`cpi` went `324.245 → NaN → 325.063` from Sep to Nov), so linear interpolation between the neighboring months is a reasonable, low-risk fix. It is much safer than for `total_price`, since macro data doesn't have the sharp jumps individual transactions can have.

`.interpolate()` fills each NaN by drawing a straight line between the nearest valid values before and after it (needs the rows sorted by date, which `macro` already is):

In [15]:
macro[['cpi', 'real_wage_index']] = macro[['cpi', 'real_wage_index']].interpolate()
macro.isnull().sum()

date                   0
cpi                    0
avg_hourly_earnings    0
real_wage_index        0
dtype: int64

### 4. Cleaning Duplicate Records

#### 4.1 `transactions`, `macro`, `weather` — confirm no duplicates

Step 2 already showed `.duplicated().sum()` is 0 for all three tables, meaning no fully identical rows.

But I also want to check duplicates on the column(s) that should uniquely identify a real-world record). Check each table's actual key below:

In [16]:
transactions['transaction_id'].duplicated().sum()
#no duplicates

np.int64(0)

In [17]:
macro['date'].duplicated().sum()
#no duplicates

np.int64(0)

In [18]:
weather.duplicated(subset=['date', 'city']).sum()
#no duplicates

np.int64(0)

`transactions`, `macro`, and `weather` all have no duplicates to clean in this step.

#### 4.2 `menu` — resolving duplicates

Two separate problems, per the Open Decision above:
1. Step 2 found exactly 1 fully duplicated row — a straightforward drop.
2. Even after that, `menu` has 241 rows but only 106 unique `(product_name, size)` keys. As a dimension table, `menu` needs exactly one row per key.

Drop the exact duplicate first:

In [19]:
menu = menu.drop_duplicates().reset_index(drop=True)
menu.shape

(241, 8)

Now check how many rows are still duplicated on the business key `(product_name, size)`:

In [20]:
menu.duplicated(subset=['product_name', 'size']).sum()

np.int64(135)

135 rows need to collapse into their group's single representative row via `groupby(['product_name', 'size']).agg(...)`. Aggregation rule per column:

- `price_usd`, `calories`, `sugar_g`, `caffeine_mg` — **median** per group.
Per the Open Decision, these values don't need to be reconciled for the join to work, so median is just a reasonable single number to keep. This also resolves 2 of the 23 `menu.caffeine_mg` nulls from Step 3.3 for free, since `median()` skips NaNs; the other 21 raw null rows sit in groups that are null in every row, so those groups stay NULL after collapsing to one row each.
- `category` — **first** (whatever value happens to be in the group's first row). Not authoritative, but doesn't need to be per the Open Decision — a mode-based version was checked against this and produces identical results on every group anyway, so there's no reason to keep the extra complexity.
- `seasonal_flag` — it's the only `menu` column `04_data_integration.ipynb`'s join needs. Rule: majority vote per group; if `0` and `1` tie, default to `1` (True).

In [21]:
def resolve_seasonal_flag(s):
    # s.mean() is the share of 1s in the group, since values are only 0/1.
    # share >= 0.5 covers both "1 is the majority" and the tie case (share == 0.5 -> prefer True).
    return int(s.mean() >= 0.5)

resolve_seasonal_flag(pd.Series([0, 0, 1]))   # majority case -> 0

0

In [22]:
menu = menu.groupby(['product_name', 'size'], as_index=False).agg({
    'category':       'first',
    'price_usd':      'median',
    'calories':       'median',
    'sugar_g':        'median',
    'caffeine_mg':    'median',
    'seasonal_flag':  resolve_seasonal_flag,
})

menu.shape

(106, 8)

Verify: no duplicate keys left, and check how many `caffeine_mg` nulls remain.

In [23]:
print('duplicate keys:', menu.duplicated(subset=['product_name', 'size']).sum())
print('caffeine_mg nulls:', menu['caffeine_mg'].isnull().sum())

duplicate keys: 0
caffeine_mg nulls: 13


0 duplicate keys — `menu` is now 106 rows, one per `(product_name, size)`.

13 rows still have a NULL `caffeine_mg`. These are the products where every raw row for that `(product_name, size)` was null (the 21 raw null rows from Step 3.3 collapse into 13 group-level nulls once each group becomes a single row). This doesn't block anything downstream, since `menu`'s nutritional columns aren't used in the master dataset join.

This closes the **Menu dedup rule** Open Decision from the top of this notebook.

### 5. Data Type Conversion

1. `menu.calories` / `menu.sugar_g` — round to the nearest whole number and cast to `int64`, to match `transactions.calories` / `transactions.sugar_g` (already `int64`).
2. `date` columns in `transactions`, `macro`, `weather` — convert from string (`object`) to real `datetime64` dates.
3. `transactions.is_weekend` / `menu.seasonal_flag` — keep as `int64` (0/1), no conversion.

#### 5.1 `menu.calories` / `menu.sugar_g` → round to `int64`

These two only became `float64` in Step 4 because `median()` on an even-sized group can land exactly between two integers.

In [24]:
menu[(menu['calories'] % 1 != 0) | (menu['sugar_g'] % 1 != 0)][['product_name', 'size', 'calories', 'sugar_g']]

,product_name,size,calories,sugar_g
24,Caramel (Without Whipped Cream),Grande,270.0,57.5
35,Coffee,Grande,220.0,48.5
36,Coffee,Tall,125.0,27.5
38,Espresso,Grande,7.5,0.0


4 rows have a `.5` value. Round and cast both columns to `int64`:

In [25]:
menu[['calories', 'sugar_g']] = menu[['calories', 'sugar_g']].round().astype('int64')
menu[['calories', 'sugar_g']].dtypes

calories    int64
sugar_g     int64
dtype: object

Results:

In [26]:
menu[menu['product_name'].isin(['Espresso', 'Coffee', 'Caramel (Without Whipped Cream)'])][['product_name', 'size', 'calories', 'sugar_g']]

,product_name,size,calories,sugar_g
24,Caramel (Without Whipped Cream),Grande,270,58
25,Caramel (Without Whipped Cream),Tall,180,41
26,Caramel (Without Whipped Cream),Venti,330,77
35,Coffee,Grande,220,48
36,Coffee,Tall,125,28
37,Coffee,Venti,235,51
38,Espresso,Grande,8,0


#### 5.2 `date` columns → `datetime64`

`transactions.date`, `macro.date`, and `weather.date` are all stored as plain strings (`object` dtype). Convert each with `pd.to_datetime()` to stores it as a real date value that pandas can sort, filter, and do date arithmetic on directly:

In [27]:
transactions['date'] = pd.to_datetime(transactions['date'])
macro['date'] = pd.to_datetime(macro['date'])
weather['date'] = pd.to_datetime(weather['date'])

transactions['date'].dtype, macro['date'].dtype, weather['date'].dtype

(dtype('<M8[ns]'), dtype('<M8[ns]'), dtype('<M8[ns]'))

All three are now `datetime64[ns]`.

#### 5.3 `transactions.is_weekend` / `menu.seasonal_flag` — keep as `int64`

Decision: leave both as `int64` (0/1) rather than converting to `bool`. No code needed here.

### 6. Text Standardization

#### 6.1 General sweep — whitespace, casing, cross-table spelling

For every text column, check for values that differ from their own stripped version (stray leading/trailing whitespace) and check whether lowercasing the column collapses any two "different" values into one (a casing inconsistency, e.g. `"grande"` vs `"Grande"`):

In [28]:
text_columns = {
    'menu':         ['product_name', 'category', 'size'],
    'transactions': ['time_slot', 'city', 'persona', 'category', 'product_name', 'size', 'customizations'],
    'weather':      ['city'],
}
tables = {'menu': menu, 'transactions': transactions, 'weather': weather}

for table_name, cols in text_columns.items():
    df = tables[table_name]
    for col in cols:
        values = df[col].dropna().astype(str)
        stray_whitespace = (values != values.str.strip()).sum()
        casing_collisions = values.nunique() - values.str.lower().nunique()
        print(f'{table_name}.{col}: unique={values.nunique()}, stray_whitespace={stray_whitespace}, casing_collisions={casing_collisions}')

menu.product_name: unique=33, stray_whitespace=0, casing_collisions=0
menu.category: unique=9, stray_whitespace=0, casing_collisions=0
menu.size: unique=4, stray_whitespace=0, casing_collisions=0
transactions.time_slot: unique=6, stray_whitespace=0, casing_collisions=0
transactions.city: unique=5, stray_whitespace=0, casing_collisions=0
transactions.persona: unique=5, stray_whitespace=0, casing_collisions=0
transactions.category: unique=9, stray_whitespace=0, casing_collisions=0
transactions.product_name: unique=33, stray_whitespace=0, casing_collisions=0
transactions.size: unique=4, stray_whitespace=0, casing_collisions=0
transactions.customizations: unique=116, stray_whitespace=0, casing_collisions=0
weather.city: unique=5, stray_whitespace=0, casing_collisions=0


Every column comes back `stray_whitespace=0, casing_collisions=0`. Now check that `menu` and `transactions` actually agree on the exact spelling of the columns they share. If one table had a typo the other didn't, they'd fail to join on that value later:

In [29]:
for col in ['product_name', 'category', 'size']:
    only_in_menu = set(menu[col]) - set(transactions[col])
    only_in_transactions = set(transactions[col]) - set(menu[col])
    print(f'{col}: only in menu = {only_in_menu}, only in transactions = {only_in_transactions}')

product_name: only in menu = set(), only in transactions = set()
category: only in menu = set(), only in transactions = set()
size: only in menu = set(), only in transactions = set()


All are empty sets. So `menu` and `transactions` already agree on every spelling. The general sweep is clean.

#### 6.2 Deferred from Step 3.2

In 3.2, some `(product_name, size)` groups met a problem: with the same `(product_name, size)`, there are multiple `caffeine_mg` value, which makes no sense.

First, find which `(product_name, size)` groups in `transactions` have more than 1 distinct `caffeine_mg` value:

In [30]:
caffeine_nunique = transactions.groupby(['product_name', 'size'])['caffeine_mg'].nunique()
conflict_keys = caffeine_nunique[caffeine_nunique >= 2].index
list(conflict_keys)

[('Coffee', 'Grande'), ('Coffee', 'Tall'), ('Coffee', 'Venti')]

All 3 conflicts are `product_name == 'Coffee'`, just at different sizes. Check whether `caffeine_mg` is actually consistent once `category` is added to the grouping:

In [31]:
transactions[transactions['product_name'] == 'Coffee'].groupby(['size', 'category'])['caffeine_mg'].agg(['nunique', 'unique'])

nunique   unique
size   category                                           
Grande Frappuccino® Blended Coffee              1   [70.0]
       Frappuccino® Light Blended Coffee        1   [95.0]
Tall   Frappuccino® Blended Coffee              1    [0.0]
       Frappuccino® Light Blended Coffee        1   [70.0]
Venti  Frappuccino® Blended Coffee              1   [95.0]
       Frappuccino® Light Blended Coffee        1  [120.0]

`nunique` is 1 in every `(size, category)` cell. So the problem is that `"Coffee"` is being used as the `product_name` for two genuinely different drinks (`Frappuccino® Blended Coffee` and `Frappuccino® Light Blended Coffee`), and only `category` tells them apart.

Confirm no other `product_name` has this same problem:

In [32]:
category_nunique = transactions.groupby('product_name')['category'].nunique()
category_nunique[category_nunique > 1]

product_name
Coffee    2
Name: category, dtype: int64

Only `Coffee`.

**Decision:** rename `product_name` to be specific for these rows.

#### 6.3 Apply the rename

Start with `transactions`, since every raw row is still intact there — this is a straightforward row-level rename, no data loss:

In [33]:
coffee_mask = transactions['product_name'] == 'Coffee'
transactions.loc[coffee_mask, 'product_name'] = (
    transactions.loc[coffee_mask, 'product_name'] + ' (' + transactions.loc[coffee_mask, 'category'] + ')'
)

transactions.loc[coffee_mask, 'product_name'].value_counts()

product_name
Coffee (Frappuccino® Blended Coffee)          3934
Coffee (Frappuccino® Light Blended Coffee)    1009
Name: count, dtype: int64

`transactions` now has two distinct product names instead of one ambiguous one. But this exposes a real problem in `menu`: Step 4 deduped `menu` by grouping on `(product_name, size)` only — `category` wasn't part of that key. So `menu`'s 3 `Coffee` rows (one per size) each silently blended the two real products together; their `price_usd`/`calories`/`seasonal_flag`/etc. are medians/votes computed across rows that shouldn't have been grouped as one product in the first place. Confirm that's still true right now:

In [34]:
menu[menu['product_name'] == 'Coffee']

,product_name,size,category,price_usd,calories,sugar_g,caffeine_mg,seasonal_flag
35,Coffee,Grande,Frappuccino® Blended Coffee,4.870,220,48,95.0,0
36,Coffee,Tall,Frappuccino® Blended Coffee,4.285,125,28,35.0,0
37,Coffee,Venti,Frappuccino® Blended Coffee,5.335,235,51,107.5,0


3 blended rows, exactly as expected. Since `transactions` now names these rows `"Coffee (Frappuccino® Blended Coffee)"` / `"Coffee (Frappuccino® Light Blended Coffee)"`, `menu` no longer has a matching `product_name` for either one.

Fix it by re-running the same Step 4 dedup logic, but only on the `Coffee` rows after applying rename first:

In [35]:
menu_coffee_raw = pd.read_csv('../data/raw/starbucks_menu.csv').drop_duplicates()
menu_coffee_raw = menu_coffee_raw[menu_coffee_raw['product_name'] == 'Coffee'].copy()
menu_coffee_raw['product_name'] = menu_coffee_raw['product_name'] + ' (' + menu_coffee_raw['category'] + ')'

menu_coffee_fixed = menu_coffee_raw.groupby(['product_name', 'size'], as_index=False).agg({
    'category':      'first',
    'price_usd':     'median',
    'calories':      'median',
    'sugar_g':       'median',
    'caffeine_mg':   'median',
    'seasonal_flag': resolve_seasonal_flag,
})
menu_coffee_fixed[['calories', 'sugar_g']] = menu_coffee_fixed[['calories', 'sugar_g']].round().astype('int64')

menu_coffee_fixed

,product_name,size,category,price_usd,calories,sugar_g,caffeine_mg,seasonal_flag
0,Coffee (Frappuccino® Blended Coffee),Grande,Frappuccino® Blended Coffee,4.92,220,50,95.0,0
1,Coffee (Frappuccino® Blended Coffee),Tall,Frappuccino® Blended Coffee,4.33,160,36,0.0,0
2,Coffee (Frappuccino® Blended Coffee),Venti,Frappuccino® Blended Coffee,5.43,310,69,95.0,0
3,Coffee (Frappuccino® Light Blended Coffee),Grande,Frappuccino® Light Blended Coffee,4.68,120,26,95.0,0
4,Coffee (Frappuccino® Light Blended Coffee),Tall,Frappuccino® Light Blended Coffee,4.24,90,19,70.0,0
5,Coffee (Frappuccino® Light Blended Coffee),Venti,Frappuccino® Light Blended Coffee,5.24,160,33,120.0,0


6 correct rows (3 sizes × 2 real products), where there were 3 blended ones before. Swap them into `menu`drop the old blended `Coffee` rows, add these back in:

In [36]:
menu = pd.concat([
    menu[menu['product_name'] != 'Coffee'],
    menu_coffee_fixed
], ignore_index=True)

menu.shape

(109, 8)

106 → 109 rows (3 blended rows removed, 6 correct ones added: 109 = 106 - 3 + 6). Verify no more bare `"Coffee"` in either table, and the two tables' `product_name` vocabularies line up again:

In [37]:
print('bare "Coffee" left in menu:', (menu['product_name'] == 'Coffee').sum())
print('bare "Coffee" left in transactions:', (transactions['product_name'] == 'Coffee').sum())
print()
only_in_menu = set(menu['product_name']) - set(transactions['product_name'])
only_in_transactions = set(transactions['product_name']) - set(menu['product_name'])
print('only in menu:', only_in_menu)
print('only in transactions:', only_in_transactions)

bare "Coffee" left in menu: 0
bare "Coffee" left in transactions: 0

only in menu: set()
only in transactions: set()


0 bare `"Coffee"` rows left in either table, and both `only in menu` / `only in transactions` are empty sets — the two tables are fully realigned on `product_name`.

### 7. Cleaning Invalid / Impossible Values

In [38]:
#check negatives
for name, df in [('menu', menu), ('transactions', transactions), ('macro', macro), ('weather', weather)]:
    for col in df.select_dtypes('number').columns:
        neg = (df[col] < 0).sum()
        if neg:
            print(f'{name}.{col}: {neg} negative values')

transactions.temp_f: 130 negative values
weather.temp_mean_f: 4 negative values
weather.temp_min_f: 10 negative values


Only temperature columns show negatives, which is reasonable. No negative prices/nutrition/counts anywhere.

In [39]:
parsed_count = transactions['customizations'].apply(lambda s: 0 if s == 'none' else len(s.split('|')))
print('n_customizations mismatches:', (parsed_count != transactions['n_customizations']).sum())

actual_weekend = transactions['date'].dt.dayofweek >= 5
print('is_weekend mismatches:', (actual_weekend != transactions['is_weekend'].astype(bool)).sum())

print('weather temp_min<=mean<=max violations:', ((weather['temp_min_f'] > weather['temp_mean_f']) | (weather['temp_mean_f'] > weather['temp_max_f'])).sum())

print(transactions.groupby('time_slot')['hour'].agg(['min', 'max']))

n_customizations mismatches: 0
is_weekend mismatches: 0
weather temp_min<=mean<=max violations: 0
               min  max
time_slot              
afternoon       14   16
early_morning    5    6
evening         17   20
late_morning    10   11
lunch           12   13
morning_rush     7    9


`hour` ranges per `time_slot` don't overlap. All logically consistent.

In [40]:
ratio = transactions['total_price'] / (transactions['base_price'] + transactions['upcharge'])
ratio.corr(transactions['cpi'])

np.float64(0.999243196123894)

0.999 correlation — `total_price` is `(base_price+upcharge)` scaled by a CPI-linked factor. Explains Step 3.1's mystery. Not a defect.

### 8. Outlier

### 9. Cleaning Summary & Cleaned Dataset